In [2]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.
# Generative AI tools (including coding assistants) are not permitted for this assignment.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** XX  
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line):

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

In [8]:
tickers = pd.read_csv('Tickers_Example.csv', header=None, names = ['Ticker'])

#created to filter all tickers based on assignment requirements. Returns a list of tickers that meet the requirements.
def filter_ticks (tickers, min_volume = 5000, min_days_per_month = 18):
    start_date = '2024-11-27'
    end_date = '2026-11-26'
    filtered = []
    close_dictionary = {}
    currencies = {}
    sectors = {}

    for i in tickers['Ticker'].drop_duplicates(): #passes ticker column of the csv, dropping duplicates to avoid them appearing twice
        try:
            ticker = yf.Ticker(i)
            history = ticker.history(start=start_date, end=end_date)
            
            if (history.empty): #ticker~ is invalid or has no data for the specified date range
                continue

            vol = history['Volume']
            month = history.index.strftime('%Y-%m')  #masks to group by month
            avg_vol = vol[vol.groupby(month).transform('count') >= min_days_per_month] #filters data by making sure the number of days is greater than 18

            
            if ((avg_vol.mean() >= min_volume) or avg_vol.empty): #verifies stock has a daily volume of at least 5000 shares over the past year
                info = ticker.info #saves runtime by running this after the initial check
                market = info.get('market')
                if (market == 'us_market' or market == 'ca_market') and (info.get('currency') == 'USD' or info.get('currency') == 'CAD') and (info.get('quoteType') == 'EQUITY'): #verifies stock is listed on a US or Canadian exchange and is traded in USD or CAD
                    filtered.append(i)
                    close = history['Close']
                    close.index = close.index.tz_localize(None).normalize() #removes timezone from index and normalizes to midnight
                    close_dictionary[i] = close
                    currencies[i] = info.get('currency')
                    sectors[i] = info.get('sector')
        except:
            continue #in case ticker is invalid, ignore the current ticker

    return filtered, close_dictionary, currencies, pd.Series(sectors, name='Sector')
                    

filtered_tickers, close_dictionary, currencies, sectors = filter_ticks(tickers)

def get_usd_cad_rate(): #converts price from USD to CAD
    start_date = '2024-11-27'
    end_date = '2026-11-26'
    rate = yf.Ticker('USDCAD=X').history(start=start_date, end=end_date)['Close']
    rate.index = rate.index.tz_localize(None).normalize()
    return rate

def get_close_prices(close_dictionary, start_date = '2024-11-27', end_date = '2026-11-26'): #change start date to 2024-11-27 and end to 2026-11-26
    close_prices = pd.DataFrame(close_dictionary)
    rate = get_usd_cad_rate()
    for ticker in close_prices.columns:
        try:
            if yf.Ticker(ticker).fast_info.get('currency') == 'USD': #if price is listed in USD
                close_prices[ticker] = rate.reindex(close.index).ffill() * close
        except:
            continue

    sp500 = yf.download('^GSPC', start=start_date, end=end_date, progress=False)['Close']['^GSPC'] #gets S&P500 close prices in USD
    close_prices['S&P500'] = sp500 * rate.reindex(sp500.index).ffill() #adds S&P500 to the dataframe in CAD
    close_prices['Conversion_Rate'] = rate #adds conversion rate to the dataframe
    return close_prices

all_close_prices = get_close_prices(close_dictionary)
print(all_close_prices)


'''returns a dataframe of monthly returns given a ticker and the close prices dataframe.'''
def get_monthly_returns(ticker, close_prices):
    monthly_returns = close_prices[ticker].resample('ME').ffill().pct_change().dropna() #resample to monthly frequency, forward fill missing values, calculate percentage change, and drop NaN values
    return monthly_returns

#print(get_sectors(filtered_tickers))

$AGN: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1d 2024-11-27 -> 2026-11-26)
$CELG: possibly delisted; no timezone found
$MON: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found


                  AAPL        ABBV         ABT         ACN        AIG  \
Date                                                                    
2024-11-27  233.060303  171.754379  113.906876  347.068695  73.688126   
2024-11-28         NaN         NaN         NaN         NaN        NaN   
2024-11-29  235.441208  171.613632  113.734528  347.269928  73.439751   
2024-12-02  237.683228  170.525406  111.857620  346.321228  72.608688   
2024-12-03  240.718842  170.272110  111.359657  337.849548  71.739403   
...                ...         ...         ...         ...        ...   
2026-11-19  349.209991  283.540009  103.870003  218.580002  79.500000   
2026-11-20  342.559998  286.200012  103.000000  221.169998  79.680000   
2026-11-23  349.079987  288.450012  103.320000  222.289993  80.559998   
2026-11-24  354.070007  300.279999  105.389999  232.320007  81.239998   
2026-11-25  355.579987  294.480011  108.000000  240.490005  82.230003   

                  AMZN         AXP          BA    

## Client A — Growth

Construct a portfolio intended to achieve the **highest expected return**. Use your selected stocks and portfolio weights to create a DataFrame named `Portfolio_Final_A` with index **1 through n** and columns (in this order): `Ticker`, `Price`, `Currency`, `Shares`, `Value`, `Weight`. `Price` is in the stock’s quoted currency, `Value` is in CAD, and `Weight` is a fraction summing to 1.

In [ ]:
budget_a = 1_000_000
number_of_stocks_a = 10
start_date = '2024-11-27'
end_date = '2026-11-27'
max_stock_weight = 0.15
max_sector_weight = 0.40
max_fee = 2.15
fee_per_share = 0.001


exchange_rate = get_usd_cad_rate()

sectors = {}
currencies = {}
buy_price_cad = {}

for ticker in filtered_tickers:
    try:
        info = yf.Ticker(ticker).info
    except Exception:
        continue
    if ticker in all_close_prices.columns and not pd.isna(all_close_prices[ticker].iloc[-1]):
        sectors[ticker] = info.get('sector', 'Unknown')
        currencies[ticker] = info.get('currency')
        buy_price_cad[ticker] = all_close_prices[ticker].iloc[-1]


monthly_closes = {}

for ticker in list(buy_price_cad) + ['^GSPC', 'USDCAD=X']:
    close = yf.Ticker(ticker).history(start=start_date, end=end_date)['Close']
    close.index = close.index.tz_localize(None)
    monthly_closes[ticker] = close.resample('ME').last()

monthly_prices = pd.DataFrame(monthly_closes)

for ticker in monthly_prices.columns:
    if currencies.get(ticker) == 'USD' or ticker == '^GSPC':
        monthly_prices[ticker] = monthly_prices[ticker] * monthly_prices['USDCAD=X']

monthly_returns = monthly_prices.pct_change().iloc[1:]
market_returns = monthly_returns['^GSPC']


rows = []

for ticker in buy_price_cad:
    expected_return = monthly_returns[ticker].mean()
    beta = monthly_returns[ticker].cov(market_returns) / market_returns.var()
    if not pd.isna(expected_return) and not pd.isna(beta):
        rows.append({'Ticker': ticker, 'Sector': sectors[ticker],
                     'Expected_Return': expected_return, 'Beta': beta})

stats_a = pd.DataFrame(rows).set_index('Ticker')
stats_a['Rank_ER'] = stats_a['Expected_Return'].rank(ascending=False)
stats_a['Rank_Beta'] = stats_a['Beta'].rank(ascending=False)
stats_a['Score'] = stats_a['Rank_ER'] + stats_a['Rank_Beta']
stats_a = stats_a.sort_values(['Score', 'Expected_Return'], ascending=[True, False])


max_per_sector = int(max_sector_weight * number_of_stocks_a)
picked_a = []
sector_count = {}

for ticker in stats_a.index:
    if len(picked_a) == number_of_stocks_a:
        break
    sector = stats_a.loc[ticker, 'Sector']
    if stats_a.loc[ticker, 'Expected_Return'] > 0 and sector_count.get(sector, 0) < max_per_sector:
        picked_a.append(ticker)
        sector_count[sector] = sector_count.get(sector, 0) + 1

expected = {ticker: stats_a.loc[ticker, 'Expected_Return'] for ticker in picked_a}

weights_a = {}
for ticker in picked_a:
    weights_a[ticker] = expected[ticker] / sum(expected.values())

for i in range(1000):
    extra = 0

    for ticker in picked_a:
        if weights_a[ticker] > max_stock_weight:
            extra += weights_a[ticker] - max_stock_weight
            weights_a[ticker] = max_stock_weight

    for sector in set(sectors[ticker] for ticker in picked_a):
        in_sector = [ticker for ticker in picked_a if sectors[ticker] == sector]
        sector_total = sum(weights_a[ticker] for ticker in in_sector)
        if sector_total > max_sector_weight:
            for ticker in in_sector:
                new_weight = weights_a[ticker] * max_sector_weight / sector_total
                extra += weights_a[ticker] - new_weight
                weights_a[ticker] = new_weight

    if extra == 0:
        break

    has_room = []
    for ticker in picked_a:
        sector_total = sum(weights_a[t] for t in picked_a if sectors[t] == sectors[ticker])
        if weights_a[ticker] < max_stock_weight and sector_total < max_sector_weight:
            has_room.append(ticker)

    room_total = sum(expected[ticker] for ticker in has_room)
    for ticker in has_room:
        weights_a[ticker] += extra * expected[ticker] / room_total


shares_a = {}

money_for_stocks = budget_a

for i in range(100):

    fees_a = 0
    
    for ticker in picked_a:
        shares_a[ticker] = weights_a[ticker] * money_for_stocks / buy_price_cad[ticker]
        fees_a += min(max_fee, fee_per_share * shares_a[ticker]) * exchange_rate

    new_money_for_stocks = budget_a - fees_a

    if abs(new_money_for_stocks - money_for_stocks) < 1e-9:
        break

    money_for_stocks = new_money_for_stocks


rows = []

for ticker in picked_a:
    price = buy_price_cad[ticker]
    if currencies[ticker] == 'USD':
        price = buy_price_cad[ticker] / exchange_rate
    rows.append({'Ticker': ticker, 'Price': price, 'Currency': currencies[ticker],
                 'Shares': shares_a[ticker], 'Value': shares_a[ticker] * buy_price_cad[ticker]})

Portfolio_Final_A = pd.DataFrame(rows)
Portfolio_Final_A['Weight'] = Portfolio_Final_A['Value'] / Portfolio_Final_A['Value'].sum()
Portfolio_Final_A = Portfolio_Final_A.sort_values('Value', ascending=False)
Portfolio_Final_A.index = range(1, len(Portfolio_Final_A) + 1)

purchases_a = Portfolio_Final_A['Value'].sum()

print(Portfolio_Final_A)


     Ticker        Price Currency        Shares          Value    Weight
1     BB.TO    12.070000      CAD  12427.415549  149998.901888  0.150000
2     TD.TO   164.710007      CAD    875.686755  144234.371314  0.144235
3       LLY  1392.219971      USD     62.269821  121830.948364  0.121832
4         C   113.330002      USD    645.777288  102848.937570  0.102850
5      BIIB   261.440002      USD    264.282974   97098.690159  0.097099
6       CAT   543.690002      USD    121.705532   92989.478518  0.092990
7       TXN   270.190002      USD    240.586334   91350.804784  0.091351
8     RY.TO   269.260010      CAD    326.382213   87881.677993  0.087882
9      QCOM   168.660004      USD    266.666153   63205.103646  0.063206
10  SHOP.TO   189.669998      CAD    255.990750   48553.765015  0.048554


### Client A — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

### Client B — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

## Contribution and AI Declaration

**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.